# AI Laboratory: Bayesian Networks and Autoregressive Language Models

This notebook implements the laboratory specification using the six-sentence dataset provided in the lab sheet. It contains:

- a first-order autoregressive language model, `P(X_t | X_{t-1})`;
- conditional probability tables (CPTs);
- probability-normalisation tests;
- next-word prediction for multiple contexts;
- sampling-based and greedy text generation;
- a second-order model, `P(X_t | X_{t-2}, X_{t-1})`;
- a comparison of the two models.

Only ordinary Python data structures and random sampling are used; no machine-learning library or pretrained model is used.


## 1. Dataset and tokenisation

In [13]:
sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

tokenised_sentences = [
    ["<START>"] + sentence.lower().split() + ["<END>"]
    for sentence in sentences
]

for sentence in tokenised_sentences:
    print(sentence)


['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']


## 2. First-order model: transition counts and CPT

In [14]:
from collections import Counter, defaultdict
import random

first_order_counts = defaultdict(Counter)

for sentence in tokenised_sentences:
    for current_word, next_word in zip(sentence, sentence[1:]):
        first_order_counts[current_word][next_word] += 1


def normalise_counts(count_table):
    probabilities = {}
    for context, next_counts in count_table.items():
        total = sum(next_counts.values())
        probabilities[context] = {
            word: count / total for word, count in next_counts.items()
        }
    return probabilities


first_order_probabilities = normalise_counts(first_order_counts)

print("Transition counts:")
for word, counts in first_order_counts.items():
    print(f"{word:>8} -> {dict(counts)}")


Transition counts:
 <START> -> {'the': 6}
     the -> {'cat': 3, 'mat': 2, 'rug': 2, 'dog': 3, 'park': 2}
     cat -> {'sat': 2, 'ran': 1}
     sat -> {'on': 4}
      on -> {'the': 4}
     mat -> {'<END>': 2}
     rug -> {'<END>': 2}
     dog -> {'sat': 2, 'ran': 1}
     ran -> {'to': 2}
      to -> {'the': 2}
    park -> {'<END>': 2}


### Selected conditional probability distributions (Question 3)

In [15]:
selected_words = ["the", "cat", "dog", "sat", "ran"]

for word in selected_words:
    print(f"P(next word | {word}) =")
    for next_word, probability in first_order_probabilities[word].items():
        print(f"  {next_word:>7}: {probability:.6f}")
    print()

print("Any transition not listed for a context has maximum-likelihood probability 0.")


P(next word | the) =
      cat: 0.250000
      mat: 0.166667
      rug: 0.166667
      dog: 0.250000
     park: 0.166667

P(next word | cat) =
      sat: 0.666667
      ran: 0.333333

P(next word | dog) =
      sat: 0.666667
      ran: 0.333333

P(next word | sat) =
       on: 1.000000

P(next word | ran) =
       to: 1.000000

Any transition not listed for a context has maximum-likelihood probability 0.


## 3. Probability-normalisation test

In [16]:
def check_normalisation(probabilities, tolerance=1e-12):
    all_valid = True
    for context, distribution in probabilities.items():
        total = sum(distribution.values())
        ok = abs(total - 1.0) <= tolerance
        all_valid = all_valid and ok
        print(f"{str(context):>20} : total = {total:.12f}  {'PASS' if ok else 'FAIL'}")
    return all_valid

print("First-order normalisation test:")
first_order_normalised = check_normalisation(first_order_probabilities)
print("All first-order distributions normalised:", first_order_normalised)


First-order normalisation test:
             <START> : total = 1.000000000000  PASS
                 the : total = 1.000000000000  PASS
                 cat : total = 1.000000000000  PASS
                 sat : total = 1.000000000000  PASS
                  on : total = 1.000000000000  PASS
                 mat : total = 1.000000000000  PASS
                 rug : total = 1.000000000000  PASS
                 dog : total = 1.000000000000  PASS
                 ran : total = 1.000000000000  PASS
                  to : total = 1.000000000000  PASS
                park : total = 1.000000000000  PASS
All first-order distributions normalised: True


## 4. Next-word prediction

In [17]:
def most_probable_next(distribution):
    if not distribution:
        return None
    return max(distribution, key=distribution.get)

prediction_contexts = ["the", "cat", "dog", "sat", "ran", "on", "to"]

for context in prediction_contexts:
    distribution = first_order_probabilities.get(context, {})
    print(f"Context: {context!r}")
    print(" Distribution:", distribution)
    print(" Most probable next word:", most_probable_next(distribution))
    print()


Context: 'the'
 Distribution: {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
 Most probable next word: cat

Context: 'cat'
 Distribution: {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
 Most probable next word: sat

Context: 'dog'
 Distribution: {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
 Most probable next word: sat

Context: 'sat'
 Distribution: {'on': 1.0}
 Most probable next word: on

Context: 'ran'
 Distribution: {'to': 1.0}
 Most probable next word: to

Context: 'on'
 Distribution: {'the': 1.0}
 Most probable next word: the

Context: 'to'
 Distribution: {'the': 1.0}
 Most probable next word: the



## 5. First-order text generation

In [18]:
def sample_from_distribution(distribution, rng):
    words = list(distribution.keys())
    weights = list(distribution.values())
    if not words:
        return None
    return rng.choices(words, weights=weights, k=1)[0]


def generate_first_order(probabilities, mode="sampling", seed=None, max_tokens=30):
    rng = random.Random(seed)
    current = "<START>"
    output = []

    for _ in range(max_tokens):
        distribution = probabilities.get(current)
        if not distribution:
            return " ".join(output), "stopped: unseen/no-outgoing context"

        if mode == "greedy":
            next_word = most_probable_next(distribution)
        elif mode == "sampling":
            next_word = sample_from_distribution(distribution, rng)
        else:
            raise ValueError("mode must be 'greedy' or 'sampling'")

        if next_word == "<END>":
            return " ".join(output), "ended normally"

        output.append(next_word)
        current = next_word

    return " ".join(output), f"truncated at {max_tokens} tokens"


print("20 first-order sampled sentences:")
first_order_samples = []
for i in range(20):
    sentence, status = generate_first_order(
        first_order_probabilities, mode="sampling", seed=100 + i
    )
    first_order_samples.append(sentence)
    print(f"{i+1:2d}. {sentence} [{status}]")


20 first-order sampled sentences:
 1. the rug [ended normally]
 2. the cat ran to the dog sat on the dog sat on the cat sat on the rug [ended normally]
 3. the dog sat on the mat [ended normally]
 4. the rug [ended normally]
 5. the park [ended normally]
 6. the mat [ended normally]
 7. the park [ended normally]
 8. the rug [ended normally]
 9. the dog ran to the cat sat on the park [ended normally]
10. the rug [ended normally]
11. the mat [ended normally]
12. the cat ran to the dog sat on the cat ran to the mat [ended normally]
13. the dog sat on the dog sat on the mat [ended normally]
14. the park [ended normally]
15. the cat sat on the cat ran to the cat ran to the cat ran to the cat ran to the mat [ended normally]
16. the cat sat on the park [ended normally]
17. the dog ran to the dog sat on the mat [ended normally]
18. the cat sat on the cat sat on the dog sat on the cat sat on the dog sat on the park [ended normally]
19. the cat sat on the mat [ended normally]
20. the cat ran to 

## 6. Greedy versus sampling generation

In [19]:
print("Five first-order greedy generations:")
for i in range(5):
    sentence, status = generate_first_order(
        first_order_probabilities, mode="greedy", seed=i, max_tokens=24
    )
    print(f"{i+1}. {sentence} [{status}]")

print("\nFive first-order sampling generations:")
for i in range(5):
    sentence, status = generate_first_order(
        first_order_probabilities, mode="sampling", seed=200 + i, max_tokens=24
    )
    print(f"{i+1}. {sentence} [{status}]")


Five first-order greedy generations:
1. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on [truncated at 24 tokens]
2. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on [truncated at 24 tokens]
3. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on [truncated at 24 tokens]
4. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on [truncated at 24 tokens]
5. the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on [truncated at 24 tokens]

Five first-order sampling generations:
1. the cat ran to the mat [ended normally]
2. the mat [ended normally]
3. the dog sat on the park [ended normally]
4. the park [ended normally]
5. the cat sat on the mat [ended normally]


**Observation.** Sampling produces more variation because each next token is drawn according to its probability. Greedy generation repeatedly chooses the highest-probability transition. In this small first-order model, greedy generation can enter a loop because `the -> cat` (or `dog`) is more probable than the sentence-ending branches through `mat`, `rug`, or `park`; therefore a maximum-token safety limit is useful.


## 7. Second-order model

In [20]:
second_order_counts = defaultdict(Counter)

for sentence in tokenised_sentences:
    for i in range(2, len(sentence)):
        context = (sentence[i - 2], sentence[i - 1])
        next_word = sentence[i]
        second_order_counts[context][next_word] += 1

second_order_probabilities = normalise_counts(second_order_counts)

print("Second-order conditional distributions:")
for context, distribution in second_order_probabilities.items():
    print(f"{context} -> {distribution}")


Second-order conditional distributions:
('<START>', 'the') -> {'cat': 0.5, 'dog': 0.5}
('the', 'cat') -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
('cat', 'sat') -> {'on': 1.0}
('sat', 'on') -> {'the': 1.0}
('on', 'the') -> {'mat': 0.5, 'rug': 0.5}
('the', 'mat') -> {'<END>': 1.0}
('the', 'rug') -> {'<END>': 1.0}
('the', 'dog') -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
('dog', 'sat') -> {'on': 1.0}
('dog', 'ran') -> {'to': 1.0}
('ran', 'to') -> {'the': 1.0}
('to', 'the') -> {'park': 1.0}
('the', 'park') -> {'<END>': 1.0}
('cat', 'ran') -> {'to': 1.0}


### Second-order normalisation test

In [21]:
print("Second-order normalisation test:")
second_order_normalised = check_normalisation(second_order_probabilities)
print("All second-order distributions normalised:", second_order_normalised)


Second-order normalisation test:
  ('<START>', 'the') : total = 1.000000000000  PASS
      ('the', 'cat') : total = 1.000000000000  PASS
      ('cat', 'sat') : total = 1.000000000000  PASS
       ('sat', 'on') : total = 1.000000000000  PASS
       ('on', 'the') : total = 1.000000000000  PASS
      ('the', 'mat') : total = 1.000000000000  PASS
      ('the', 'rug') : total = 1.000000000000  PASS
      ('the', 'dog') : total = 1.000000000000  PASS
      ('dog', 'sat') : total = 1.000000000000  PASS
      ('dog', 'ran') : total = 1.000000000000  PASS
       ('ran', 'to') : total = 1.000000000000  PASS
       ('to', 'the') : total = 1.000000000000  PASS
     ('the', 'park') : total = 1.000000000000  PASS
      ('cat', 'ran') : total = 1.000000000000  PASS
All second-order distributions normalised: True


## 8. Second-order text generation

In [22]:
def generate_second_order(first_probs, second_probs, mode="sampling", seed=None, max_tokens=30):
    rng = random.Random(seed)

    # Bootstrap the first real token from P(X1 | <START>).
    start_distribution = first_probs.get("<START>", {})
    if not start_distribution:
        return "", "stopped: no <START> distribution"

    if mode == "greedy":
        first_word = most_probable_next(start_distribution)
    elif mode == "sampling":
        first_word = sample_from_distribution(start_distribution, rng)
    else:
        raise ValueError("mode must be 'greedy' or 'sampling'")

    if first_word == "<END>":
        return "", "ended normally"

    output = [first_word]
    previous_previous = "<START>"
    previous = first_word

    for _ in range(max_tokens - 1):
        context = (previous_previous, previous)
        distribution = second_probs.get(context)
        if not distribution:
            return " ".join(output), f"stopped: unseen context {context}"

        if mode == "greedy":
            next_word = most_probable_next(distribution)
        else:
            next_word = sample_from_distribution(distribution, rng)

        if next_word == "<END>":
            return " ".join(output), "ended normally"

        output.append(next_word)
        previous_previous, previous = previous, next_word

    return " ".join(output), f"truncated at {max_tokens} tokens"


print("Five second-order greedy generations:")
for i in range(5):
    sentence, status = generate_second_order(
        first_order_probabilities, second_order_probabilities,
        mode="greedy", seed=i
    )
    print(f"{i+1}. {sentence} [{status}]")

print("\nTen second-order sampling generations:")
second_order_samples = []
for i in range(10):
    sentence, status = generate_second_order(
        first_order_probabilities, second_order_probabilities,
        mode="sampling", seed=300 + i
    )
    second_order_samples.append(sentence)
    print(f"{i+1:2d}. {sentence} [{status}]")


Five second-order greedy generations:
1. the cat sat on the mat [ended normally]
2. the cat sat on the mat [ended normally]
3. the cat sat on the mat [ended normally]
4. the cat sat on the mat [ended normally]
5. the cat sat on the mat [ended normally]

Ten second-order sampling generations:
 1. the cat sat on the mat [ended normally]
 2. the dog sat on the rug [ended normally]
 3. the dog sat on the mat [ended normally]
 4. the dog sat on the mat [ended normally]
 5. the cat ran to the park [ended normally]
 6. the dog sat on the rug [ended normally]
 7. the dog sat on the rug [ended normally]
 8. the cat ran to the park [ended normally]
 9. the dog sat on the mat [ended normally]
10. the cat sat on the rug [ended normally]


## 9. First-order versus second-order comparison

In [23]:
vocabulary = sorted({token for sentence in tokenised_sentences for token in sentence})

first_nonzero_parameters = sum(len(dist) for dist in first_order_probabilities.values())
second_nonzero_parameters = sum(len(dist) for dist in second_order_probabilities.values())

# A transparent sparsity convention:
# - first-order candidate contexts are all vocabulary tokens;
# - second-order candidate contexts are all ordered vocabulary-token pairs.
first_zero_contexts = len(vocabulary) - len(first_order_probabilities)
second_zero_contexts = len(vocabulary) ** 2 - len(second_order_probabilities)

print("Vocabulary size:", len(vocabulary))
print("First-order observed contexts:", len(first_order_probabilities))
print("Second-order observed contexts:", len(second_order_probabilities))
print("First-order stored non-zero probability entries:", first_nonzero_parameters)
print("Second-order stored non-zero probability entries:", second_nonzero_parameters)
print("First-order zero-probability contexts (under stated convention):", first_zero_contexts)
print("Second-order zero-probability contexts (under stated convention):", second_zero_contexts)

print("\nUnique sampled sentences in 20 first-order samples:", len(set(first_order_samples)))
print("Unique sampled sentences in 10 second-order samples:", len(set(second_order_samples)))


Vocabulary size: 12
First-order observed contexts: 11
Second-order observed contexts: 14
First-order stored non-zero probability entries: 17
Second-order stored non-zero probability entries: 18
First-order zero-probability contexts (under stated convention): 1
Second-order zero-probability contexts (under stated convention): 130

Unique sampled sentences in 20 first-order samples: 14
Unique sampled sentences in 10 second-order samples: 5


### Qualitative comparison

- **First-order model:** only the immediately preceding word is used. This can combine locally valid transitions into globally odd or repetitive sentences. It also creates more variation.
- **Second-order model:** the previous two tokens are used. On this tiny dataset, it usually stays much closer to the observed training sentences and is more locally coherent.
- **Data sparsity:** the second-order CPT is indexed by token pairs rather than single tokens, so the space of possible contexts grows quickly. Many token pairs are never observed in this six-sentence dataset.

The zero-context counts above use an explicit counting convention over the full vocabulary context space, because the laboratory asks for this comparison but does not prescribe a unique counting convention.


## 10. Behaviour for an unseen context

In [24]:
unseen_word = "elephant"
print("First-order distribution for unseen word:", first_order_probabilities.get(unseen_word, {}))

unseen_pair = ("cat", "dog")
print("Second-order distribution for unseen pair:", second_order_probabilities.get(unseen_pair, {}))
print("The implementation handles these safely by returning an empty distribution and stopping generation.")


First-order distribution for unseen word: {}
Second-order distribution for unseen pair: {}
The implementation handles these safely by returning an empty distribution and stopping generation.


## 11. Validation note

The implementation was checked against the probabilistic specification rather than accepted only because it runs. In particular:

1. transition counts are visible and inspectable;
2. each conditional distribution is constructed by dividing each outgoing count by the total outgoing count for that context;
3. every observed conditional distribution is tested to sum to approximately `1.0`;
4. sampling mode uses weighted random sampling, while greedy mode uses `arg max`;
5. unseen contexts are handled explicitly instead of causing an uncontrolled error.
